In [ ]:
#http://www.somascape.org/midi/tech/mfile.html#delta
#https://docs.python.org/3/library/struct.html
#https://signalmidi.app/edit


In [182]:
import struct

def read_variable_length(data, offset):
    # reads a variable acording to the midi the bottom 7 bits are used for value
    # if the 8th bit is 0 the value is done if the 8th bit is 1 grabs one more byte
    value = 0
    while True:
        byte = data[offset]
        offset += 1
        value = (value << 7) | (byte & 0x7F)
        if not (byte & 0x80):
            break
    return value, offset
    
def readMidiFile(path):
    src = open(path, "rb")
    current_ticks = 0
    tempo = 500000
    ticks_per_quarter = 480
    header_dict = {}
    current_note = {}
    melody_list = []
    
    while True:
        chunkID = src.read(4)
        chunklen = int.from_bytes(src.read(4))
        chunk = src.read(chunklen)
        delta_times = False
        if chunkID == b'MThd':
            formt, ntracks, header_dict['tickdiv']  = struct.unpack('>HHH', chunk[0:6])
            header_dict['us_per_tick'] = tempo/header_dict['tickdiv'] 
            if formt != 1:
                    print('Only format 1 MIDI files are suported')
            #print('Header: ', header_dict)
        elif chunkID == b'MTrk':
            offset = 0
            #print("track:")
            #print(chunk)
            while offset < chunklen:
                # Read delta time
                delta, offset = read_variable_length(chunk, offset)
                current_ticks += delta
                event_status = chunk[offset]
                offset += 1
                if event_status == 0xFF: #Meta event
                    meta_type = chunk[offset]
                    offset += 1
                    meta_length = chunk[offset]
                    offset += 1
                    meta_data = chunk[offset:offset+meta_length]
                    offset += meta_length
                    if meta_type == 0x2F and meta_data == b'':
                        pass
                        #print(current_ticks,"- Metadata > End of track")
                        
                    elif meta_type == 0x03:
                        pass
                        #print(current_ticks,"- Metadata > Track/Sequence name > ",meta_data)
                    elif meta_type == 0x51:
                        tempo = struct.unpack('>i', b'\x00'+meta_data)[0]
                        header_dict['us_per_tick'] = tempo/header_dict['tickdiv']
                        header_dict['bpm'] = 60000000/tempo
                        #print(current_ticks,"- Metadata > Tempo > ",tempo)
                    elif meta_type == 0x58:
                        num, denom, clocks_per_clck, notes_per_beat = struct.unpack('>bbbb', meta_data)
                        #print(current_ticks,"- Metadata > Time signature >",num, denom, clocks_per_clck, notes_per_beat)
                    else:
                        pass
                        #print(current_ticks,'- type - ',meta_type,' - Meta Data > ',meta_data)
                elif event_status//16 == 9 or event_status//16 == 8:
                    note = chunk[offset]
                    offset += 1
                    vel = chunk[offset]
                    offset += 1
                    event_time = int(current_ticks*header_dict['us_per_tick'])
                    if event_status//16 == 9:
                        note_ac = 'Note On'
                        current_note = {'n':note,'st':current_ticks}
                    else:
                        note_ac = 'Note Off'
                        current_note['delt'] = current_ticks-current_note['st']
                        melody_list.append(current_note)
                    
                    #print(note_ac+' > n:',note,' v:',vel,'t:',event_time)
                elif event_status//16 == 11:
                    controller = chunk[offset]
                    offset += 1
                    value = chunk[offset]
                    offset += 1
                    #print(current_ticks,' - controller > ctrl: ',controller,'val: ',value)
                elif event_status//16 == 12:
                    pgrm_n = chunk[offset]
                    offset += 1
                    #print(current_ticks,' - program > pgrm_n: ',pgrm_n)
                elif event_status//16 == 14:
                    lsb = chunk[offset]
                    offset += 1
                    msb = chunk[offset]
                    offset += 1
                    #print(current_ticks,' - pitchBend > lsb: ',lsb,' msb: ',msb)
                else:
                    pass
                    #print(current_ticks,' - ',hex(event_status))
        else:
            return({'header':header_dict,'melody':melody_list})

In [183]:
mid = readMidiFile('../ChipTune Melodies/Kid Dracula - Go-Go at the Great Castle.mid')
mid

{'header': {'tickdiv': 480, 'µs_per_tick': 1041.6666666666667, 'bpm': 120.0},
 'melody': [{'n': 85, 'st': 720, 'delt': 240},
  {'n': 83, 'st': 960, 'delt': 240},
  {'n': 81, 'st': 1200, 'delt': 240},
  {'n': 76, 'st': 1440, 'delt': 720},
  {'n': 81, 'st': 2400, 'delt': 240},
  {'n': 83, 'st': 2640, 'delt': 480},
  {'n': 85, 'st': 3120, 'delt': 480},
  {'n': 81, 'st': 3600, 'delt': 480},
  {'n': 86, 'st': 4080, 'delt': 720},
  {'n': 85, 'st': 4800, 'delt': 720},
  {'n': 82, 'st': 5520, 'delt': 480},
  {'n': 83, 'st': 6000, 'delt': 720},
  {'n': 85, 'st': 6960, 'delt': 240},
  {'n': 83, 'st': 7200, 'delt': 240},
  {'n': 81, 'st': 7440, 'delt': 240},
  {'n': 76, 'st': 7680, 'delt': 720},
  {'n': 81, 'st': 8400, 'delt': 240},
  {'n': 83, 'st': 8640, 'delt': 480},
  {'n': 85, 'st': 9120, 'delt': 480},
  {'n': 88, 'st': 9600, 'delt': 480},
  {'n': 86, 'st': 10080, 'delt': 960},
  {'n': 88, 'st': 11040, 'delt': 960},
  {'n': 90, 'st': 12000, 'delt': 240},
  {'n': 90, 'st': 12480, 'delt': 720}